# 01 - Data

Question: what is in this table, and what exactly are we predicting?

Source: Rain in Australia (Kaggle), `../data.raw/weatherAUS.csv`. One row is one day at one weather station.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data.raw/weatherAUS.csv', parse_dates=['Date'])
print(df.shape)
print(df.head(3).T)

(145460, 23)
                                 0                    1                    2
Date           2008-12-01 00:00:00  2008-12-02 00:00:00  2008-12-03 00:00:00
Location                    Albury               Albury               Albury
MinTemp                       13.4                  7.4                 12.9
MaxTemp                       22.9                 25.1                 25.7
Rainfall                       0.6                  0.0                  0.0
Evaporation                    NaN                  NaN                  NaN
Sunshine                       NaN                  NaN                  NaN
WindGustDir                      W                  WNW                  WSW
WindGustSpeed                 44.0                 44.0                 46.0
WindDir9am                       W                  NNW                    W
WindDir3pm                     WNW                  WSW                  WSW
WindSpeed9am                  20.0                  4.0        

In [3]:
print(df.dtypes.value_counts())
print(df['Date'].min(), df['Date'].max())
print(df['Location'].nunique())
print(df.duplicated(['Date', 'Location']).sum())

float64           16
str                6
datetime64[us]     1
Name: count, dtype: int64
2007-11-01 00:00:00 2017-06-25 00:00:00
49
0


In [4]:
print(df['RainTomorrow'].value_counts(dropna=False))
print(df['RainTomorrow'].value_counts(dropna=False, normalize=True).round(3))

RainTomorrow
No     110316
Yes     31877
NaN      3267
Name: count, dtype: int64
RainTomorrow
No     0.758
Yes    0.219
NaN    0.022
Name: proportion, dtype: float64


In [5]:
df = df.dropna(subset=['RainTomorrow'])
print(len(df))
print(df['RainTomorrow'].value_counts(normalize=True).round(3))

142193
RainTomorrow
No     0.776
Yes    0.224
Name: proportion, dtype: float64


In [6]:
missing  = df.isna().mean().sort_values(ascending=False)
print(missing.round(3))

Sunshine         0.477
Evaporation      0.428
Cloud3pm         0.402
Cloud9am         0.377
Pressure9am      0.099
Pressure3pm      0.098
WindDir9am       0.070
WindGustDir      0.066
WindGustSpeed    0.065
WindDir3pm       0.027
Humidity3pm      0.025
Temp3pm          0.019
WindSpeed3pm     0.018
Humidity9am      0.012
Rainfall         0.010
RainToday        0.010
WindSpeed9am     0.009
Temp9am          0.006
MinTemp          0.004
MaxTemp          0.002
Location         0.000
Date             0.000
RainTomorrow     0.000
dtype: float64


In [7]:
cols = ['Sunshine', 'Evaporation', 'Cloud3pm', 'Cloud9am']
by_loc = df[cols].isna().groupby(df['Location']).mean()
print(by_loc.round(2).sort_values('Sunshine'))

                  Sunshine  Evaporation  Cloud3pm  Cloud9am
Location                                                   
Darwin                0.00         0.02      0.00      0.00
Hobart                0.00         0.00      0.37      0.36
Melbourne             0.00         0.00      0.19      0.17
MelbourneAirport      0.00         0.00      0.00      0.00
Perth                 0.00         0.00      0.00      0.00
PearceRAAF            0.00         1.00      0.37      0.34
Sydney                0.00         0.02      0.17      0.17
PerthAirport          0.00         0.00      0.00      0.00
Watsonia              0.00         0.00      0.00      0.00
SydneyAirport         0.01         0.02      0.00      0.00
Brisbane              0.02         0.01      0.00      0.00
Mildura               0.04         0.04      0.05      0.03
Nuriootpa             0.05         0.04      0.25      0.04
NorfolkIsland         0.13         0.07      0.02      0.02
WaggaWagga            0.13         0.04 

In [8]:
d = df.dropna(subset=['RainTomorrow']).copy()
d['y'] = (d['RainTomorrow'] == 'Yes').astype(int)

by_station = d.groupby('Location')['y'].agg(['mean', 'size']).sort_values('mean')
print(by_station.round(3).head(5))
print(by_station.round(3).tail(5))

               mean  size
Location                 
Woomera       0.068  2990
Uluru         0.076  1521
AliceSprings  0.081  3031
Mildura       0.109  3007
Cobar         0.129  2988
                mean  size
Location                  
NorfolkIsland  0.310  2964
Dartmoor       0.313  2943
Cairns         0.318  2988
Walpole        0.337  2819
Portland       0.365  2996


In [9]:
d['Date'] = pd.to_datetime(d['Date'])
print(d.groupby(d['Date'].dt.month)['y'].mean().round(3))

Date
1     0.193
2     0.210
3     0.213
4     0.218
5     0.225
6     0.262
7     0.269
8     0.253
9     0.230
10    0.196
11    0.213
12    0.208
Name: y, dtype: float64


In [11]:
num = d.select_dtypes('number').drop(columns='y')
means = num.groupby(d['y']).mean()
gap = ((means.loc[1] - means.loc[0]) / num.std()).sort_values(key=abs, ascending=False)
print(gap.round(2))

Sunshine        -1.08
Humidity3pm      1.07
Cloud3pm         0.90
Cloud9am         0.75
Humidity9am      0.62
Pressure9am     -0.59
Rainfall         0.58
WindGustSpeed    0.56
Pressure3pm     -0.54
Temp3pm         -0.46
MaxTemp         -0.38
Evaporation     -0.29
WindSpeed9am     0.22
WindSpeed3pm     0.21
MinTemp          0.20
Temp9am         -0.06
dtype: float64


Finding: Rain rate varies far more by station (6.8% to 36.5%) than by month (19.3% to 26.9%). Sunshine, Humidity3pm and the cloud columns separate rainy from dry days best.

Why it matters: Location carries real signal, and the strongest feature is missing entirely for 19 of 49 stations.

Decision: Location as a feature. Defer the Sunshine/Evaporation/Cloud handling to notebook 04.

Limit: The gaps compare one feature at a time, so they say nothing about features that overlap (Sunshine and Cloud are probably measuring much the same thing).